# 05c · 월드 항등식과 패턴 대기시간 (Wald's Identity and Pattern Waiting Times)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Williams §10.10 (선택적 정지) · Williams Chapter 10 연습문제 (E10.x, ABRACADABRA) · G&S 4e §10.2 (Wald's equation) · §12.5 (선택적 정지의 예로서 Wald 항등식) · Durrett 3e §5.3–§5.4 (정지시각 응용) · Ross 12e §7.3 (재생 정리 증명에 쓰이는 Wald's equation) · Li (1980), Ann. Probab. 8(6) (패턴 대기시간과 Conway 공식의 마팅게일 증명) |
| 선수 노트북 | 05b (선택적 정지 정리), 01c (흡수 사슬의 기본행렬) |
| 예상 소요 | 90분 |
| 상태 | draft |

05b에서 선택적 정지 정리(OST)를 손에 넣었다. 이 노트북은 그 정리를 두 방향으로 쓴다. 첫째, i.i.d. 합 $S_n$을 정지시각 $T$에서 멈추면 **월드 항등식(Wald's identity)** $E[S_T]=E[X_1]E[T]$가 나온다 — 드리프트 걸음의 도달시각 기대값과 분산이 한 줄로 떨어진다. 둘째, 공정 동전에서 어떤 패턴(HTH, HHH, …)이 처음 나올 때까지의 기대 시각을 **도박꾼 마팅게일(ABRACADABRA 논증)**로 구한다: 답은 패턴이 자기 자신과 얼마나 겹치는가로만 결정된다(HTH는 10, HHH는 14, HHT는 8). 같은 값을 01c의 흡수 사슬(기본행렬)과 시뮬레이션으로 다시 얻어 세 방법을 맞춰 보고, 마지막에 '기대 대기시간이 짧다'와 '먼저 나온다'가 **다른 양**임을 Penney 게임의 비추이성으로 확인한다.

## 0. 준비 (Setup)

In [ ]:
import os
from itertools import product
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import absorption, is_stochastic, random_walk_paths, hitting_times

SEED, rng = rng_for("05c")
setup_plots()
N_PATHS = scale(100_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기 (패턴 대기시간 MC 경로 수)
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (05b) 선택적 정지 정리의 세 가지 충분조건을 말하라.

<details><summary>정답</summary>

(a) $T$가 유계; (b) $T<\infty$ a.s.이고 정지 과정 $M_{n\wedge T}$가 유계; (c) $E[T]<\infty$이고 $\{T>n\}$ 위에서 증분 $|M_{n+1}-M_n|$이 유계. 오늘은 거의 전부 (c)를 쓴다.

</details>

**Q2.** (05b) $p\ne1/2$인 파산 문제에서 사용한 두 마팅게일과 각각이 주는 답은?

<details><summary>정답</summary>

$(q/p)^{S_n}$ → 탈출 확률 $h(i)=(1-r^i)/(1-r^N)$, $r=q/p$; $S_n-(p-q)n$ → 기대 시간 $E_i[T]=(N\,h(i)-i)/(p-q)$. 두 번째가 오늘의 월드 항등식 $E[S_T]=\mu E[T]$ 그 자체다.

</details>

**Q3.** (05b) 배가 전략은 $E[T]=2<\infty$인데 왜 $E[M_T]=1\ne0$이 가능한가?

<details><summary>정답</summary>

OST(c)는 $E[T]<\infty$뿐 아니라 증분 유계도 요구한다. 배가 전략의 증분은 $2^n$으로 무계이고, (a)·(b)도 깨진다. 오늘 도박꾼 마팅게일에서 '증분 유계'를 확인하는 이유다.

</details>

**Q4.** (05a) 우도비 $L_n=\prod_{i\le n}f(X_i)/g(X_i)$가 $g$ 아래에서 마팅게일인 이유는?

<details><summary>정답</summary>

$E_g[f(X)/g(X)]=\sum_xf(x)=1$이므로 각 인자의 조건부 기대가 1: $E[L_{n+1}\mid\mathcal F_n]=L_n\,E_g[f(X_{n+1})/g(X_{n+1})]=L_n$. '공정한 배당의 곱'이 마팅게일이라는 같은 구조가 오늘 도박꾼의 자본 $2^k$에 나타난다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **월드 항등식(Wald's identity / equation)**: $X_1,X_2,\dots$ i.i.d., $E|X_1|<\infty$, $\mu=E[X_1]$, $S_n=\sum_{i\le n}X_i$. $T$가 $(X_i)$가 만드는 필트레이션 $\mathcal F_n=\sigma(X_1,\dots,X_n)$에 대한 정지시각이고 $E[T]<\infty$이면 $E[S_T]=\mu\,E[T]$.
- **월드 2차 항등식**: 추가로 $\sigma^2=\mathrm{Var}(X_1)<\infty$이면 $E[(S_T-\mu T)^2]=\sigma^2E[T]$.
- **패턴 대기시각(pattern waiting time)**: 공정 동전 $\xi_1,\xi_2,\dots\in\{H,T\}$, 길이 $m$ 패턴 $A=A_1\cdots A_m$에 대해 $T_A=\inf\{n\ge m:(\xi_{n-m+1},\dots,\xi_n)=A\}$ — **슬라이딩 윈도우**로 처음 완성되는 시각.
- **겹침(overlap) 집합**: $\mathrm{ov}(A)=\{k\in\{1,\dots,m\}:A_{1..k}=A_{m-k+1..m}\}$, 길이 $k$ 접두부 = 길이 $k$ 접미부. 항상 $m\in\mathrm{ov}(A)$. 예: $\mathrm{ov}(\text{HTH})=\{1,3\}$, $\mathrm{ov}(\text{HHH})=\{1,2,3\}$, $\mathrm{ov}(\text{HHT})=\{3\}$.
- **도박꾼 마팅게일(ABRACADABRA 논증)**: 매 시각 $j$에 도박꾼 $j$가 1원을 들고 입장해 $\xi_j=A_1$에 전 재산을 건다(맞으면 2배, 틀리면 파산 후 퇴장). 살아 있으면 다음 던지기에서 $A_2$에 전 재산을 걸고, … 패턴을 완성하면 $2^m$을 갖고 퇴장한다. 시각 $n$의 총 순손익(도박꾼 전체가 본 순이익) $M_n=(\text{시각 }n\text{에 살아 있는 도박꾼들의 자본 합})-n$ ($n$은 지금까지 낸 입장료 1원들).
- **KMP 자동자 사슬(pattern automaton)**: 상태 $s\in\{0,\dots,m\}$ = '지금까지 관측의 접미부 중 $A$의 접두부와 일치하는 최대 길이'. $s=m$은 흡수. 전이: $s\to$ (길이 $s$ 접두부에 다음 글자를 붙인 문자열의 최장 접미부–접두부 일치 길이). $E[T_A]$ = 상태 0에서의 기대 흡수시간 $=(N\mathbf 1)_0$, $N=(I-Q)^{-1}$ (01c). 코드: `kmp_matrix` + `absorption`.
- **Penney 게임과 Conway 상관수**: 두 패턴 $A,B$(같은 길이 $m$, 어느 쪽도 다른 쪽을 포함하지 않음). $A{:}B=\sum_{k=1}^m2^k\,[A\text{의 길이 }k\text{ 접미부}=B\text{의 길이 }k\text{ 접두부}]$ (순서가 중요: $A{:}B\ne B{:}A$ 일반). $A{:}A$는 겹침 공식 $E[T_A]$와 같다. 예: $A=$THH, $B=$HHH: $A{:}A=8$, $A{:}B=6$ (THH의 접미부 H, HH가 HHH의 접두부), $B{:}B=14$, $B{:}A=0$.

### 2.2 정리 1 — 월드 항등식 (G&S 4e §10.2, §12.5; Durrett 3e §5.3)

> $X_i$ i.i.d., $E|X_1|<\infty$, $T$ 정지시각, $E[T]<\infty$ $\Longrightarrow$ $E[S_T]=E[X_1]\,E[T]$.

**가정이 왜 필요한가**
- $E[T]<\infty$: 없으면 거짓. 대칭 걸음($\mu=0$)의 $T_1=\inf\{n:S_n=1\}$은 a.s. 유한하지만 $E[S_{T_1}]=1\ne0\cdot\infty$. 증명에서 합과 기대를 바꾸는 Fubini 조건이기도 하다.
- $T$가 **정지시각**: $\{T\ge i\}=\{T\le i-1\}^c$가 $X_1,\dots,X_{i-1}$로만 결정되어 $X_i$와 독립이어야 $E[X_i1_{\{T\ge i\}}]=\mu P(T\ge i)$가 된다. '$S_n$이 최대가 되는 시각'처럼 미래를 보고 멈추면 $E[S_T]>\mu E[T]$ (E3).
- i.i.d.와 적분가능: $\sum_iE|X_i|P(T\ge i)=E|X_1|E[T]<\infty$가 Fubini를 정당화한다. 독립이지만 동일분포가 아니면 $E[S_T]=\sum_iE[X_i]P(T\ge i)$까지만 성립한다.

<details><summary>증명</summary>

**증명 1 (직접, 더 일반적).** $S_T=\sum_{i\ge1}X_i\,1_{\{T\ge i\}}$ (시각 $T$까지 더한 항만 남는다). 사건 $\{T\ge i\}=\{T\le i-1\}^c\in\mathcal F_{i-1}$는 $X_1,\dots,X_{i-1}$의 함수이므로 $X_i$와 독립이다. 따라서
$$E[X_i1_{\{T\ge i\}}]=E[X_i]\,P(T\ge i)=\mu\,P(T\ge i).$$
합과 기대의 교환: $\sum_iE[|X_i|1_{\{T\ge i\}}]=E|X_1|\sum_iP(T\ge i)=E|X_1|\,E[T]<\infty$이므로 Fubini(또는 지배수렴)로 $E[S_T]=\sum_iE[X_i1_{\{T\ge i\}}]$. 마지막으로 꼬리합 공식 $\sum_{i\ge1}P(T\ge i)=E[T]$를 쓰면 $E[S_T]=\mu E[T]$. $\square$

**증명 2 (OST, 증분이 유계일 때).** $M_n=S_n-n\mu$는 마팅게일이다: $E[M_{n+1}-M_n\mid\mathcal F_n]=E[X_{n+1}]-\mu=0$. $|X_i|\le c$이면 $|M_{n+1}-M_n|\le c+|\mu|$로 증분이 유계이고 $E[T]<\infty$이므로 OST(c)에 의해 $E[M_T]=E[M_0]=0$, 즉 $E[S_T]-\mu E[T]=0$. $\square$

증명 1은 $X_i$가 유계가 아니어도(예: 지수분포 증분, Ross 12e §7.3의 재생 과정) 성립하므로 더 일반적이다. 증명 2는 '왜 마팅게일 장에 있는가'를 보여준다.

</details>

### 2.3 정리 2 — 월드 2차 항등식 (G&S 4e §12.5)

> 정리 1의 조건에 더해 $\sigma^2=\mathrm{Var}(X_1)<\infty$이면 $E[(S_T-\mu T)^2]=\sigma^2E[T]$. 특히 $\mu>0$, $S_T=a$(상수)인 도달 문제에서 $E[T]=a/\mu$, $\mathrm{Var}(T)=\sigma^2a/\mu^3$.

**가정이 왜 필요한가**
- $\sigma^2<\infty$: 2차 모멘트가 없으면 좌변이 정의되지 않는다.
- $E[T]<\infty$: 1차 항등식과 같은 이유이고, 직교 증분 합을 $T\wedge n$에서 절단한 뒤 극한을 넘길 때 필요하다.

<details><summary>증명 스케치</summary>

$Y_i=X_i-\mu$, $\tilde S_n=\sum_{i\le n}Y_i=S_n-n\mu$라 하자. 절단한 합 $\tilde S_{T\wedge n}=\sum_{i\le n}Y_i1_{\{T\ge i\}}$를 제곱해 전개하면 대각항은 $E[Y_i^21_{\{T\ge i\}}]=\sigma^2P(T\ge i)$ (독립성), 교차항은 $i<j$에서 $Y_i1_{\{T\ge j\}}$가 $\mathcal F_{j-1}$-가측이므로 $E[Y_iY_j1_{\{T\ge j\}}]=E[Y_i1_{\{T\ge j\}}]E[Y_j]=0$. 따라서
$$E[\tilde S_{T\wedge n}^2]=\sigma^2\sum_{i\le n}P(T\ge i)=\sigma^2E[T\wedge n].$$
같은 계산으로 $E[(\tilde S_{T\wedge n}-\tilde S_{T\wedge k})^2]=\sigma^2E[T\wedge n-T\wedge k]\to0$ ($k,n\to\infty$)이므로 $\tilde S_{T\wedge n}$은 $L^2$ 코시 열이고 극한은 $\tilde S_T$ (a.s. 수렴과 일치). $L^2$ 수렴은 2차 모멘트의 수렴을 주므로 $E[\tilde S_T^2]=\sigma^2E[T]$ [G&S 4e §12.5, Williams §10.10 참조].

**도달 문제.** $S_T=a$이면 $\tilde S_T=a-\mu T$. 정리 1로 $E[T]=a/\mu$이므로 $E[(a-\mu T)^2]=\mu^2\mathrm{Var}(T)+(a-\mu E[T])^2=\mu^2\mathrm{Var}(T)$. 이것이 $\sigma^2E[T]=\sigma^2a/\mu$와 같으므로 $\mathrm{Var}(T)=\sigma^2a/\mu^3$. $\square$

</details>

### 2.4 정리 3 — 패턴 대기시간 (ABRACADABRA; Williams §10.10 · Chapter 10 연습문제 E10.x, Li 1980)

> 공정 동전, 길이 $m$ 패턴 $A$: $$E[T_A]=\sum_{k\in\mathrm{ov}(A)}2^k.$$ 앞면 확률 $p$인 일반 동전에서는 $E[T_A]=\sum_{k\in\mathrm{ov}(A)}1/P(\xi_{1..k}=A_{1..k})$. 예: HTH $=2^3+2^1=10$, HHH $=8+4+2=14$, HHT $=8$, HT $=4$, HH $=6$, 26자 알파벳의 ABRACADABRA $=26^{11}+26^4+26$.

**가정이 왜 필요한가**
- $E[T_A]<\infty$ (OST(c)의 첫 조건): 서로 겹치지 않는 길이 $m$ 블록 각각이 $A$일 확률이 $2^{-m}$이므로 $P(T_A>km)\le(1-2^{-m})^k$ — 기하 꼬리, 모든 모멘트가 유한하다.
- 증분 유계 (OST(c)의 둘째 조건): 한 시각에 살아 있는 도박꾼은 최대 $m$명, 각 자본은 $2^m$ 이하이므로 $|M_{n+1}-M_n|\le\sum_{k\le m}2^k+1<2^{m+1}+1$.
- **공정성**: 각 베팅의 기대 손익이 0이어야(2배 배당 = 확률 $1/2$의 역수) 각 도박꾼의 자본이 마팅게일이다. 편향 동전이면 배당을 $1/p$, $1/q$로 바꾸면 같은 논증이 되고 $2^k$가 $1/P(\text{접두부}_k)$로 바뀐다.

<details><summary>증명</summary>

**(1) $M_n$은 마팅게일.** 도박꾼 $j$의 시각 $n$ 자본을 $C^{(j)}_n$이라 하자($n<j$이면 아직 입장 전이므로 $C^{(j)}_n=1$로 두고, 입장료 1원은 $-1$로 따로 센다). 매 베팅은 확률 $1/2$로 2배, $1/2$로 0이 되므로 $E[C^{(j)}_{n+1}\mid\mathcal F_n]=C^{(j)}_n$: 각 자본은 마팅게일(05a의 우도비처럼 '공정 배당의 곱'). $M_n=\sum_{j\le n}(C^{(j)}_n-1)$은 시각 $n$까지 입장한 유한 개 마팅게일의 합에서 상수를 뺀 것이므로 마팅게일이다. 정확히는 $M_{n+1}-M_n=\sum_{j\le n+1}(C^{(j)}_{n+1}-C^{(j)}_n)$ (새 도박꾼 $n+1$의 항 $C^{(n+1)}_{n+1}-1$ 포함)이고 각 항의 조건부 기대가 0이다.

**(2) OST(c).** 위 '가정이 왜 필요한가'의 두 조건이 성립하므로 $E[M_{T_A}]=E[M_0]=0$.

**(3) 시각 $T=T_A$에 살아 있는 도박꾼 세기.** 시각 $T-k+1$에 입장한 도박꾼이 아직 살아 있으려면 그가 본 $k$개 글자 $\xi_{T-k+1..T}$가 $A_{1..k}$와 같아야 한다. 그런데 $T$에 패턴이 완성되었으므로 $\xi_{T-k+1..T}=A_{m-k+1..m}$. 즉 $A_{1..k}=A_{m-k+1..m}$, 다시 말해 $k\in\mathrm{ov}(A)$일 때만 살아 있고 그의 자본은 $2^k$다. $k>m$인 도박꾼은 $T$ 이전에 패턴을 완성했어야 하므로($T$의 최소성에 모순) 없다. 따라서 $M_T=\sum_{k\in\mathrm{ov}(A)}2^k-T$이고 (2)에서
$$0=E[M_T]=\sum_{k\in\mathrm{ov}(A)}2^k-E[T_A].\qquad\square$$

**두 번째 풀이(흡수 사슬).** KMP 자동자 $P$의 일시 블록 $Q$에 대해 $E[T_A]=((I-Q)^{-1}\mathbf 1)_0$ (01c 정리 2). 4.3절에서 이 값이 겹침 공식과 $10^{-9}$ 안에서 같음을 모든 패턴에 대해 확인한다 — 마팅게일 풀이 = 기본행렬 풀이.

</details>

### 2.5 정리 4 — Conway 공식 (Penney 게임; Li 1980)

> 같은 길이의 두 패턴 $A,B$, $D=A{:}A-A{:}B+B{:}B-B{:}A$라 하면
> $$P(A\text{ before }B)=\frac{B{:}B-B{:}A}{D},\qquad E[\min(T_A,T_B)]=\frac{A{:}A\cdot B{:}B-A{:}B\cdot B{:}A}{D}.$$

**가정이 왜 필요한가**
- 두 패턴 모두 겹침 구조로만 결정된다 — 길이가 같아야 하는 것은 아니지만 한 패턴이 다른 패턴을 **포함하면** 안 된다(포함되면 $P(A\text{ before }B)\in\{0,1\}$로 게임이 자명하다; 공식은 여전히 성립한다).
- 공정 동전(2배 배당). 편향 동전이면 $2^k$를 $1/P(\text{접두부}_k)$로 바꾼다.

**증명 (인용 + 방정식 풀이).** Li (1980)의 논증: 패턴 $A$에 베팅하는 도박꾼 팀과 $B$에 베팅하는 팀을 동시에 운영하고 $T=\min(T_A,T_B)$에 OST(c)를 적용한다. 시각 $T$에 어느 패턴이 완성되었는가에 따라 각 팀의 생존 자본이 달라진다: $A$가 먼저면 $A$-팀은 $A{:}A$, $B$-팀은 $A{:}B$ ($A$의 접미부가 $B$의 접두부인 만큼 살아 있음), $B$가 먼저면 각각 $B{:}A$, $B{:}B$. 팀별 마팅게일에 $E[M_T]=0$을 쓰면 $P_A=P(A\text{ before }B)$와 $E[T]$에 대한 두 선형 방정식이 나온다:
$$E[T]=P_A\,A{:}A+(1-P_A)\,B{:}A,\qquad E[T]=P_A\,A{:}B+(1-P_A)\,B{:}B.$$
두 식을 빼면 $P_A(A{:}A-A{:}B)=(1-P_A)(B{:}B-B{:}A)$이므로 $P_A=(B{:}B-B{:}A)/D$; 이를 첫 식에 넣으면 $E[T]=(A{:}A\cdot B{:}B-A{:}B\cdot B{:}A)/D$. 예 (THH vs HHH): $P_A=(14-0)/(8-6+14-0)=7/8$, $E[T]=(8\cdot14-6\cdot0)/16=7$. 팀 논증의 세부(각 팀이 마팅게일인 이유, 증분 유계)는 정리 3과 같고 논문을 인용한다. 4.5절에서 결합 자동자 흡수 사슬의 선형계와 MC로 수치 검증한다.

### 2.6 직관

월드 항등식은 '평균 총합 = 평균 스텝 수 × 스텝당 평균'이며, 멈추는 규칙이 **미래를 엿보지 않는 한** 성립한다. 패턴 문제의 놀라움: 모든 길이-3 패턴은 장기 빈도가 $1/8$로 같은데 첫 등장의 평균 시각은 8, 10, 14로 다르다. 이유는 **뭉침(clustering)**이다. HHH는 한 번 나오면 HHHH처럼 연달아 나오기 쉬워(자기 겹침) 같은 빈도 $1/8$을 드문 뭉치로 채우므로 첫 등장이 늦다. 도박꾼 논증은 이 겹침을 '시각 $T$에 아직 살아 있는 도박꾼'으로 정확히 센다. Penney 게임의 비추이성(THH > HHT > HTT > TTH > THH; '>'는 '먼저 나올 확률이 $1/2$보다 크다')은 '먼저 나오기' 경쟁이 **상대 패턴의 접두부를 내 접미부로** 만드는 구조(상대가 완성되기 직전 상태를 가로채기)에 달려 있기 때문이며, 기대 대기시간과는 다른 양이다.

✍️ 내 말로: 'HHH가 HTH보다 오래 걸리는 이유'를 도박꾼 논증을 쓰지 않고, 뭉침이라는 말로 두 문장으로 설명해 보세요.

### 2.7 함정(traps)

1. 모든 길이-3 패턴의 장기 빈도는 같다($1/8$)지만 첫 등장 시각의 평균은 다르다(8, 10, 14). '확률이 같으니 대기시간도 같다'는 오해.
2. $E[T_A]<E[T_B]$라고 해서 $P(A\text{ before }B)>1/2$가 아니다. HTH(10) vs HHH(14)는 0.6이지만, THH(8) vs HHT(8)는 대기시간이 같은데 0.75이고, Penney 게임은 비추이적이다.
3. Wald에서 $T$는 정지시각이어야 한다. '$S_n$이 최대가 되는 시각'에 멈추면 $E[S_T]=E[\max_{k\le n}S_k]>0=\mu E[T]$ (E3).
4. $p=1/2$의 $T_a$($a>0$)에는 Wald를 쓸 수 없다: $E[T_a]=\infty$ (05b). 실제로 $E[S_{T_a}]=a\ne0=\mu\cdot(\text{무엇이든})$.
5. 시뮬레이션에서 패턴 검색은 슬라이딩 윈도우(연속)로 해야 한다. 겹치지 않는 블록으로 나누면 대기시간의 분포가 달라진다(예: 블록 방식의 HH 대기시간 기대값은 $2\cdot4=8\ne6$).
6. 겹침 집합에 $k=m$(패턴 전체)은 항상 포함된다. HHT의 $E[T]=8$은 '겹침 없음'이 아니라 $\mathrm{ov}=\{3\}$.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요) — (1) 월드 항등식의 진술과 세 가정, (2) 도박꾼 논증에서 '시각 $T$에 살아 있는 도박꾼'이 왜 겹침 집합과 일대일인지, (3) $P(A\text{ before }B)$와 $E[T_A]<E[T_B]$가 왜 다른 질문인지를 각각 두 문장으로.

## 3. Predict — 코드를 돌리기 전에 예측

`None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향(어느 쪽이 큰가)이 맞는지가 목적입니다. 예측을 먼저 적어 두고 4절을 실행하세요.

In [ ]:
predictions = {
    # 공정 동전에서 HTH 가 처음 나올 때까지의 기대 던지기 수는?
    "ET_HTH": None,
    # HHH 는? (HTH 보다 길까 짧을까?)
    "ET_HHH": None,
    # HHT 는?
    "ET_HHT": None,
    # p=0.6 인 걸음(+1 확률 0.6, -1 확률 0.4)이 +5 에 처음 도달하는 시각의 기대값은?
    "wald_ET_a5": None,
    # 같은 도달시각 T 의 분산은? (자릿수만이라도)
    "wald_VarT_a5": None,
    # THH 와 HHH 중 THH 가 먼저 나올 확률은?
    "penney_THH_vs_HHH": None,
    # HTH vs HHH 에서 HTH 가 먼저 나올 확률은? (HTH 의 기대 대기시간이 더 짧다는 것을 기억하며)
    "penney_HTH_vs_HHH": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 helper 함수 (spkit에 없어서 여기서 정의)

패턴을 0/1 배열(1 = H)로 두고, 각 경로에서 패턴이 처음 완성되는 시각을 **슬라이딩 윈도우**로 찾는다(함정 5). `overlap_set`/`overlap_formula`는 정리 3의 겹침 공식이다.

In [ ]:
def first_occurrence(flips, pattern):
    """helper (spkit에 없어서 여기서 정의): 경로별로 pattern(0/1 배열, 길이 m)이 처음 완성되는
    시각 T (슬라이딩 윈도우). 못 찾으면 n_flips + 1 (호출자가 절단 없음을 assert)."""
    pattern = np.asarray(pattern, dtype=np.int8)
    n_paths, n_flips = flips.shape
    m = len(pattern)
    W = n_flips - m + 1                                  # 윈도우 시작 위치 개수
    match = np.ones((n_paths, W), dtype=bool)
    for j in range(m):
        match &= flips[:, j:W + j] == pattern[j]
    found = match.any(axis=1)
    T = match.argmax(axis=1) + m                          # 첫 일치 윈도우의 끝 시각
    T[~found] = n_flips + 1
    return T

def to_bits(pattern_str):                                 # "HTH" -> [1, 0, 1]
    return np.array([1 if c == "H" else 0 for c in pattern_str], dtype=np.int8)

def overlap_set(pattern_str):                             # ov(A) = {k : 접두부_k == 접미부_k}
    m = len(pattern_str)
    return [k for k in range(1, m + 1) if pattern_str[:k] == pattern_str[m - k:]]

def overlap_formula(pattern_str, p_head=0.5):             # 정리 3: sum over ov(A) of 1/P(접두부_k)
    total = 0.0
    for k in overlap_set(pattern_str):
        total += 1 / np.prod([p_head if c == "H" else 1 - p_head for c in pattern_str[:k]])
    return float(total)

두 번째 helper 묶음: KMP 자동자 전이행렬(2.1 정의)과 Conway 상관수·공식(정리 4).

In [ ]:
def kmp_matrix(pattern_str, p_head=0.5):
    """helper: 패턴 자동자 전이행렬 (m+1)x(m+1). 상태 s = 현재 일치 길이, s = m 흡수."""
    m = len(pattern_str)
    P = np.zeros((m + 1, m + 1))
    for s in range(m):
        for c, pc in (("H", p_head), ("T", 1 - p_head)):
            w = pattern_str[:s] + c
            nxt = max([L for L in range(1, min(len(w), m) + 1) if w[-L:] == pattern_str[:L]], default=0)
            P[s, nxt] += pc
    P[m, m] = 1.0
    return P

def expected_wait_chain(pattern_str, p_head=0.5):        # E[T_A] = (N 1)_0, 01c
    m = len(pattern_str)
    return float(absorption(kmp_matrix(pattern_str, p_head), list(range(m)), [m])["t"][0])

def corr(X, Y):                                           # Conway 상관수 X:Y (공정 동전)
    return sum(2**k for k in range(1, min(len(X), len(Y)) + 1) if X[-k:] == Y[:k])

def conway_prob(A, B):                                    # P(A before B)
    D = corr(A, A) - corr(A, B) + corr(B, B) - corr(B, A)
    return (corr(B, B) - corr(B, A)) / D

def conway_wait(A, B):                                    # E[min(T_A, T_B)]
    D = corr(A, A) - corr(A, B) + corr(B, B) - corr(B, A)
    return (corr(A, A) * corr(B, B) - corr(A, B) * corr(B, A)) / D

세 번째 helper 묶음: 도달시간 정리 $P(T_a=n)=\frac an P(S_n=a)$의 pmf(4.4절)와, Penney 게임의 결합 자동자(상태 = 마지막 $m-1$글자)를 정확히 푸는 선형계(4.5절). 바로 한 쌍에 적용해 Conway 공식과 맞는지 본다.

In [ ]:
def hitting_pmf_exact(a, p, n_max):
    """helper: 도달시간 정리 P(T_a = n) = (a/n) P(S_n = a), n = 1..n_max (n ≡ a mod 2 에서만 > 0)."""
    n = np.arange(1, n_max + 1)
    ok = ((n + a) % 2 == 0) & (n >= a)
    return n, np.where(ok, (a / n) * binom.pmf(np.where(ok, (n + a) // 2, 0), n, p), 0.0)

def penney_exact(A, B):
    """helper: 결합 자동자(상태 = 마지막 m-1 글자 이하)의 선형계로 P(A first), E[min(T_A, T_B)] 를 정확히 푼다."""
    m = max(len(A), len(B))
    states = [""] + ["".join(t) for L in range(1, m) for t in product("HT", repeat=L)]
    idx = {s: i for i, s in enumerate(states)}
    M, rA = np.eye(len(states)), np.zeros(len(states))      # (I - Q) x = r
    for s in states:
        for c in "HT":
            w = s + c
            if w.endswith(A):
                rA[idx[s]] += 0.5
            elif not w.endswith(B):
                M[idx[s], idx[w[-(m - 1):]]] -= 0.5
    return float(np.linalg.solve(M, rA)[0]), float(np.linalg.solve(M, np.ones(len(states)))[0])

결합 자동자 풀이를 한 쌍에 적용해 Conway 공식(정리 4)과 맞는지 바로 본다.

In [ ]:
print("penney_exact(THH, HHH) =", penney_exact("THH", "HHH"), "  Conway:", conway_prob("THH", "HHH"), conway_wait("THH", "HHH"))

### 4.2 패턴 대기시간 MC (step 1)

`N_PATHS`개의 경로에 300번씩 동전을 던진다(1 = H). 먼저 경로 몇 개를 그려 $T_{HTH}$, $T_{HHH}$가 무엇인지 눈으로 본다. 300번 안에 HHH를 못 찾을 확률은 $P(T_{HHH}>300)\approx10^{-11}$ (자동자 $Q$의 최대 고유값 $0.9196^{300}$)이므로 절단이 없어야 하고, 이를 assert한다.

In [ ]:
N_FLIPS = 300
flips = rng.integers(0, 2, size=(N_PATHS, N_FLIPS), dtype=np.int8)     # 1 = H, 0 = T

PATTERNS = ["HH", "HT", "HHH", "HHT", "HTH", "HTT", "THH", "THT", "TTH", "TTT"]
T_pat = {pat: first_occurrence(flips, to_bits(pat)) for pat in PATTERNS}
for pat, T in T_pat.items():
    assert np.all(T <= N_FLIPS), f"{pat}: 300 던지기 안에 못 찾은 경로가 있음 (절단)"
print("all", len(PATTERNS), "patterns found within", N_FLIPS, "flips on every path")

fig, ax = plt.subplots(figsize=(9, 3))
n_show = 40
for k in range(3):
    ax.scatter(np.arange(1, n_show + 1), np.full(n_show, k), c=flips[k, :n_show], cmap="coolwarm", vmin=0, vmax=1, s=45)
    ax.plot([T_pat["HTH"][k]] * 2, [k - 0.3, k + 0.3], color="C2", lw=3)
    ax.plot([T_pat["HHH"][k]] * 2, [k - 0.3, k + 0.3], color="k", lw=3, ls=":")
ax.set_yticks(range(3)); ax.set_yticklabels([f"path {k}" for k in range(3)]); ax.set_xlabel("toss n")
ax.set_xlim(0, n_show + 1); ax.set_title("Sample coin tosses (red = H, blue = T): green bar = T_HTH, dotted = T_HHH")
plt.show()

각 패턴의 MC 평균 $\pm$ SE와 겹침 공식을 나란히 놓는다. 표준편차도 출력한다 — HHH의 sd(≈11.9)가 HTH(≈7.6)보다 큰 것도 뭉침의 흔적이다.

In [ ]:
est_pat = {pat: mc_estimate(T_pat[pat]) for pat in PATTERNS}
exact_pat = {pat: overlap_formula(pat) for pat in PATTERNS}
print(f"{'pattern':>8} {'ov(A)':>10} {'MC mean ± SE':>22} {'sd':>6} {'formula':>8}")
for pat in PATTERNS:
    print(f"{pat:>8} {str(overlap_set(pat)):>10} {est_pat[pat].mean:>12.3f} ± {est_pat[pat].se:<7.3f} "
          f"{T_pat[pat].std(ddof=1):>6.2f} {exact_pat[pat]:>8.1f}")

In [ ]:
pats3 = [p for p in PATTERNS if len(p) == 3]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(pats3))
axes[0].bar(x, [est_pat[p].mean for p in pats3], yerr=[1.96 * est_pat[p].se for p in pats3], capsize=4, alpha=0.6, label="MC mean (95% CI)")
axes[0].plot(x, [exact_pat[p] for p in pats3], "kD", label="martingale formula $\\sum_{k\\in ov} 2^k$")
axes[0].set_xticks(x); axes[0].set_xticklabels(pats3); axes[0].set_ylabel("E[T]")
axes[0].set_title("E[T] for the 8 patterns of length 3"); axes[0].legend()
bins = np.arange(2.5, 60.5, 1)
axes[1].hist(T_pat["HTH"], bins=bins, density=True, alpha=0.5, label="T_HTH")
axes[1].hist(T_pat["HHH"], bins=bins, density=True, alpha=0.5, label="T_HHH")
axes[1].axvline(exact_pat["HTH"], color="C0", ls="--", label=f"E[T_HTH] = {exact_pat['HTH']:.0f}")
axes[1].axvline(exact_pat["HHH"], color="C1", ls="--", label=f"E[T_HHH] = {exact_pat['HHH']:.0f}")
axes[1].set_xlabel("first-occurrence time T"); axes[1].set_ylabel("density"); axes[1].legend()
axes[1].set_title("Histogram of T_HTH and T_HHH (fair coin)")
plt.tight_layout(); plt.show()

### 4.3 두 번째 풀이: KMP 자동자 흡수 사슬 (step 2)

난수가 없는 절이다. 정리 3의 겹침 공식과 01c의 기본행렬 $t=N\mathbf 1$이 **정확히** 같은 값을 주는지를 14개 패턴, 세 가지 $p$에 대해 `assert`한다. 먼저 HTH의 자동자를 눈으로 확인한다: 상태 2(HT까지 일치)에서 T가 나오면 0으로, 상태 3(HTH)에서 흡수.

In [ ]:
P_HTH = kmp_matrix("HTH")
assert is_stochastic(P_HTH)
print("KMP automaton for HTH (rows/cols = match length 0..3):")
print(P_HTH)

P_STAR = (np.sqrt(5) - 1) / 2                     # 황금비의 역수 (E1(c) 에서 유도)
ALL_PATTERNS = PATTERNS + ["HTHT", "HHTT", "HTTH", "HHHH"]
for p_head in (0.5, 0.6, P_STAR):
    for pat in ALL_PATTERNS:
        t_chain, t_formula = expected_wait_chain(pat, p_head), overlap_formula(pat, p_head)
        assert abs(t_chain - t_formula) < 1e-9, (pat, p_head, t_chain, t_formula)
print(f"absorbing chain == overlap formula for all {len(ALL_PATTERNS)} patterns, p in {{0.5, 0.6, {P_STAR:.6f}}}")

편향 동전에서는 $2^k$가 $1/P(\text{접두부}_k)$로 바뀐다: HTH는 $1/(p^2q)+1/p$, HHH는 $1/p^3+1/p^2+1/p$. $p=0.6$에서 두 값은 8.611, 9.074; $p^*=(\sqrt5-1)/2$에서 같아진다.

In [ ]:
rows_p = []
for p_head in (0.5, 0.6, P_STAR):
    q = 1 - p_head
    rows_p.append(f"| {p_head:.6f} | {expected_wait_chain('HTH', p_head):.4f} | {1/(p_head**2*q) + 1/p_head:.4f} "
                  f"| {expected_wait_chain('HHH', p_head):.4f} | {1/p_head**3 + 1/p_head**2 + 1/p_head:.4f} |")
Markdown("| p | E[T_HTH] (chain) | 1/(p²q)+1/p | E[T_HHH] (chain) | 1/p³+1/p²+1/p |\n|---|---|---|---|---|\n" + "\n".join(rows_p))

In [ ]:
p_grid = np.linspace(0.3, 0.9, 121)
fig, ax = plt.subplots()
ax.plot(p_grid, [expected_wait_chain("HTH", p) for p in p_grid], label="E[T_HTH] = 1/(p^2 q) + 1/p")
ax.plot(p_grid, [expected_wait_chain("HHH", p) for p in p_grid], label="E[T_HHH] = 1/p^3 + 1/p^2 + 1/p")
ax.axvline(P_STAR, color="k", ls="--", lw=1, label=f"p* = (sqrt5 - 1)/2 = {P_STAR:.3f}")
ax.axhline(expected_wait_chain("HTH", P_STAR), color="gray", lw=0.8, ls=":")
ax.set_xlabel("p (head probability)"); ax.set_ylabel("expected waiting time"); ax.set_ylim(0, 40); ax.legend()
ax.set_title("E[T_HTH] and E[T_HHH] vs p (from the automaton)")
plt.show()

### 4.4 월드 항등식: 드리프트 걸음의 도달시각 (step 3)

$p=0.6$ 걸음($\mu=0.2$, $\sigma^2=4pq=0.96$)이 $a=5$에 처음 닿는 시각 $T_5$. $S_{T_5}=5$이므로 Wald는 $E[T_5]=a/\mu=25$, 2차 항등식은 $\mathrm{Var}(T_5)=\sigma^2a/\mu^3=600$을 예측한다. 지평 800: $P(T_5>800)\approx9\times10^{-10}$ (도달시간 정리 pmf의 꼬리로 계산).

In [ ]:
P_UP, A_LEVEL, N_STEPS_RW = 0.6, 5, 800
N_RW = scale(10_000, minimum=2_000)      # 분산 추정의 SE 는 4차 모멘트가 지배하므로 FAST 에서도 2,000 경로 이상
mu, sigma2 = 2 * P_UP - 1, 4 * P_UP * (1 - P_UP)
ET5_exact, VarT5_exact = A_LEVEL / mu, sigma2 * A_LEVEL / mu**3      # Wald 1차, 2차

walks = random_walk_paths(N_RW, N_STEPS_RW, rng, p=P_UP)
T5 = hitting_times(walks, A_LEVEL)
n_pmf, pmf_T5 = hitting_pmf_exact(A_LEVEL, P_UP, N_STEPS_RW)
print(f"exact P(T_5 > {N_STEPS_RW}) = {1 - pmf_T5.sum():.1e};  simulated fraction not hit: {np.isinf(T5).mean()}")
assert np.all(np.isfinite(T5))

fig, ax = plt.subplots()
for k in range(8):
    stop = int(T5[k])
    ax.plot(range(stop + 1), walks[k, :stop + 1], lw=1, alpha=0.8)
    ax.plot(stop, A_LEVEL, "k|", ms=10)
ax.axhline(A_LEVEL, color="k", lw=0.8)
ax.set_xlabel("step n"); ax.set_ylabel("S_n"); ax.set_title("Walk with p = 0.6 stopped at T_5 (tick = hitting time)")
plt.show()

측정: 표본평균, 표본분산(SE는 제곱편차의 `mc_estimate`로 — 분산 추정의 SE는 $T$의 4차 중심모멘트가 지배하고 $T_5$의 꼬리가 무거워 FAST 모드에서는 이 SE 자체가 과소추정될 수 있다; 전체 모드가 믿을 만하다), 그리고 **Wald 역산** — $E[S_{T_5}]$는 정확히 5이므로 $E[T]$의 MC 추정을 $\mu$로 곱하면 5 근처여야 한다.

In [ ]:
est_T5 = mc_estimate(T5)
est_VarT5 = mc_estimate((T5 - T5.mean())**2)
print(f"E[T_5]   MC = {est_T5}   Wald a/mu = {ET5_exact:.3f}")
print(f"Var(T_5) MC = {est_VarT5}   Wald 2nd sigma^2 a/mu^3 = {VarT5_exact:.3f}")
print(f"Wald check: mu * E[T] hat = {mu * est_T5.mean:.4f}  vs  E[S_T] = a = {A_LEVEL}")
print(f"pmf check: mean {np.sum(n_pmf * pmf_T5):.4f}, var {np.sum(n_pmf**2 * pmf_T5) - np.sum(n_pmf * pmf_T5)**2:.3f}")

fig, ax = plt.subplots()
ax.hist(T5, bins=np.arange(0.5, 150.5, 1), density=True, alpha=0.5, label="MC histogram of T_5 (unit bins)")
odd = pmf_T5[:150] > 0                                   # pmf 는 n ≡ a (mod 2) 인 n 에서만 양수
ax.plot(n_pmf[:150][odd], pmf_T5[:150][odd], "C3.-", lw=1, ms=4, label="exact pmf (5/n) P(S_n = 5), odd n")
ax.axvline(ET5_exact, color="k", ls="--", label=f"E[T] = {ET5_exact:.0f}")
ax.axvspan(ET5_exact - np.sqrt(VarT5_exact), ET5_exact + np.sqrt(VarT5_exact), color="gray", alpha=0.15, label=f"± sd = {np.sqrt(VarT5_exact):.1f}")
ax.set_xlabel("hitting time T_5"); ax.set_ylabel("probability"); ax.legend()
ax.set_title("Hitting time of +5 for the p = 0.6 walk: histogram vs exact pmf")
plt.show()

(pmf는 $n\equiv a\pmod 2$인 홀수 $n$에서만 양수이므로 그 점들만 이어 그렸다. 폭 1 히스토그램의 막대 높이는 각 $n$의 확률 그 자체이고, 짝수 $n$의 막대는 정확히 0이다.)

### 4.5 Penney 게임 (step 4)

4.2절의 `flips`를 재사용한다. 두 패턴이 같은 시각에 완성될 수는 없으므로(길이가 같고 다른 패턴) $T_A<T_B$의 비율이 곧 $P(A\text{ before }B)$다. Conway 공식과 결합 자동자의 정확한 풀이를 나란히 둔다.

In [ ]:
PAIRS = [("THH", "HHH"), ("HTH", "HHH"), ("THH", "HHT"), ("HHT", "HTT"), ("HTT", "TTH"), ("TTH", "THH")]
est_penney, exact_penney = {}, {}
print(f"{'A vs B':>12} {'MC P(A first)':>22} {'Conway':>8} {'joint chain':>12} {'E[min T] Conway':>16} {'chain':>8}")
for A, B in PAIRS:
    est_penney[(A, B)] = mc_proportion(T_pat[A] < T_pat[B])
    exact_penney[(A, B)] = conway_prob(A, B)
    p_joint, t_joint = penney_exact(A, B)
    assert abs(p_joint - exact_penney[(A, B)]) < 1e-12 and abs(t_joint - conway_wait(A, B)) < 1e-9
    print(f"{A + ' vs ' + B:>12} {str(est_penney[(A, B)]):>22} {exact_penney[(A, B)]:>8.4f} {p_joint:>12.4f} "
          f"{conway_wait(A, B):>16.4f} {t_joint:>8.4f}")

뒤의 네 쌍이 비추이 순환이다: THH가 HHT를, HHT가 HTT를, HTT가 TTH를, TTH가 THH를 각각 확률 $>1/2$로 이긴다. '가장 좋은 패턴'은 없다 — 상대가 고르면 언제나 그것을 이기는 패턴이 있다.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(PAIRS))
ax.bar(x, [exact_penney[pr] for pr in PAIRS], alpha=0.4, label="Conway formula")
ax.errorbar(x, [est_penney[pr].mean for pr in PAIRS], yerr=[1.96 * est_penney[pr].se for pr in PAIRS], fmt="ko", capsize=4, label="MC (95% CI)")
ax.axhline(0.5, color="gray", ls=":")
ax.set_xticks(x); ax.set_xticklabels([f"{A} vs {B}" for A, B in PAIRS]); ax.set_ylabel("P(A before B)"); ax.set_ylim(0, 1)
ax.annotate("intransitive cycle: THH > HHT > HTT > TTH > THH", xy=(3.5, 0.92), ha="center")
ax.set_title("Penney's game: P(A before B), MC vs Conway"); ax.legend(loc="lower left")
plt.show()

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 전부 코드로 계산했다: 패턴은 `overlap_formula`(정리 3, 4.3절에서 흡수 사슬과 일치 확인), Wald는 $a/\mu$와 $\sigma^2a/\mu^3$(정리 1·2), Penney는 `conway_prob`(정리 4, 결합 자동자와 일치 확인).

In [ ]:
rows = [
    {"name": "E[T_HTH]", "predicted": predictions["ET_HTH"], "estimate": est_pat["HTH"], "exact": exact_pat["HTH"]},
    {"name": "E[T_HHH]", "predicted": predictions["ET_HHH"], "estimate": est_pat["HHH"], "exact": exact_pat["HHH"]},
    {"name": "E[T_HHT]", "predicted": predictions["ET_HHT"], "estimate": est_pat["HHT"], "exact": exact_pat["HHT"]},
    {"name": "E[T_HH]", "predicted": None, "estimate": est_pat["HH"], "exact": exact_pat["HH"]},
    {"name": "E[T_5], p=0.6 (Wald)", "predicted": predictions["wald_ET_a5"], "estimate": est_T5, "exact": ET5_exact},
    {"name": "Var(T_5), p=0.6 (Wald 2nd)", "predicted": predictions["wald_VarT_a5"], "estimate": est_VarT5, "exact": VarT5_exact},
    {"name": "P(THH before HHH)", "predicted": predictions["penney_THH_vs_HHH"], "estimate": est_penney[("THH", "HHH")], "exact": exact_penney[("THH", "HHH")]},
    {"name": "P(HTH before HHH)", "predicted": predictions["penney_HTH_vs_HHH"], "estimate": est_penney[("HTH", "HHH")], "exact": exact_penney[("HTH", "HHH")]},
    {"name": "P(THH before HHT)", "predicted": None, "estimate": est_penney[("THH", "HHT")], "exact": exact_penney[("THH", "HHT")]},
    {"name": "P(HHT before HTT)", "predicted": None, "estimate": est_penney[("HHT", "HTT")], "exact": exact_penney[("HHT", "HTT")]},
    {"name": "P(TTH before THH)", "predicted": None, "estimate": est_penney[("TTH", "THH")], "exact": exact_penney[("TTH", "THH")]},
]
Markdown(compare_table(rows))

- **E[T_HTH], E[T_HHH], E[T_HHT], E[T_HH]**: 불일치하면 `first_occurrence`가 슬라이딩 윈도우가 아니거나(함정 5), 겹침 집합을 잘못 셌거나, 절단이 있다는 뜻.
- **E[T_5]**: 불일치하면 Wald의 가정 중 하나가 깨진 것 — 지평 절단($E[T]<\infty$처럼 보이게 만드는 절단은 평균을 **과소**추정) 또는 정지 규칙이 미래를 봄.
- **Var(T_5)**: 불일치하면 2차 항등식의 $\sigma^2=4pq$를 잘못 넣었거나(자주 하는 실수: $\sigma^2=1$), 절단.
- **Penney 확률들**: 불일치하면 Conway 상관수의 방향($A{:}B$ = $A$의 접미부 vs $B$의 접두부)을 뒤집었거나, 동시 완성(길이가 다를 때 가능)을 처리하지 않은 것.

In [ ]:
assert_close(est_pat["HTH"], exact_pat["HTH"], k=4, name="E[T_HTH]")
assert_close(est_pat["HHH"], exact_pat["HHH"], k=4, name="E[T_HHH]")
assert_close(est_pat["HHT"], exact_pat["HHT"], k=4, name="E[T_HHT]")
assert_close(est_pat["HH"], exact_pat["HH"], k=4, name="E[T_HH]")
assert_close(est_T5, ET5_exact, k=4, name="Wald E[T_5]")
assert_close(est_VarT5, VarT5_exact, k=4, name="Wald 2nd Var(T_5)")
assert_close(est_penney[("THH", "HHH")], exact_penney[("THH", "HHH")], k=4, name="Penney THH vs HHH")
assert_close(est_penney[("HTH", "HHH")], exact_penney[("HTH", "HHH")], k=4, name="Penney HTH vs HHH")
assert_close(est_penney[("THH", "HHT")], exact_penney[("THH", "HHT")], k=4, name="Penney THH vs HHT")
assert_close(est_penney[("HHT", "HTT")], exact_penney[("HHT", "HTT")], k=4, name="Penney HHT vs HTT")
assert_close(est_penney[("TTH", "THH")], exact_penney[("TTH", "THH")], k=4, name="Penney TTH vs THH")
print("all MC-vs-exact checks passed (k=4)")

## 6. 연습문제

### E1. 계산

(a) 공정 동전에서 HHTT, HTHT, HTTH, HHHH의 $E[T]$를 겹침 집합으로 구하라.
(b) 26자 알파벳에서 무작위로 타자를 치는 원숭이가 ABRACADABRA를 처음 완성하는 기대 시각을 구하라.
(c) 앞면 확률 $p$인 동전에서 $E[T_{HTH}]$와 $E[T_{HHH}]$를 $p$의 식으로 쓰고, 두 값이 같아지는 $p$를 구하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $\mathrm{ov}(\text{HHTT})=\{4\}$: $16$. $\mathrm{ov}(\text{HTHT})=\{2,4\}$: $4+16=20$. $\mathrm{ov}(\text{HTTH})=\{1,4\}$: $2+16=18$. $\mathrm{ov}(\text{HHHH})=\{1,2,3,4\}$: $2+4+8+16=30$.

(b) $\mathrm{ov}(\text{ABRACADABRA})=\{1,4,11\}$ (A, ABRA, 전체). 알파벳 크기 26이므로 $2^k$ 대신 $26^k$: $E[T]=26+26^4+26^{11}=3{,}670{,}344{,}487{,}444{,}778$ (약 $3.67\times10^{15}$).

(c) $E[T_{HTH}]=\dfrac1{p^2q}+\dfrac1p$, $E[T_{HHH}]=\dfrac1{p^3}+\dfrac1{p^2}+\dfrac1p$. 같게 두고 $1/p$를 지우면 $\dfrac1{p^2q}=\dfrac1{p^3}+\dfrac1{p^2}$, 양변에 $p^3q$를 곱하면 $p=q+pq=(1-p)(1+p)=1-p^2$, 즉 $p^2+p-1=0$, $p^*=(\sqrt5-1)/2\approx0.618034$ (황금비의 역수). 공통값은 $8.472136$. $p>p^*$이면 HHH가 더 빠르다.

```python
for pat in ["HHTT", "HTHT", "HTTH", "HHHH"]:
    print(pat, overlap_set(pat), overlap_formula(pat))          # 16, 20, 18, 30
print(sum(26**k for k in overlap_set("ABRACADABRA")))           # 3670344487444778
p_star = (np.sqrt(5) - 1) / 2
print(p_star, overlap_formula("HTH", p_star), overlap_formula("HHH", p_star))   # 0.618034, 8.472136, 8.472136
```

</details>

### E2. 유도 후 시뮬레이션 검증

2차 월드 항등식으로 $p=0.7$ 걸음이 $+10$에 처음 도달하는 시각 $T_{10}$의 기대값과 분산을 유도하고, 10,000개 경로(300 스텝)로 검증하라. 지평 300이 충분한지도 근거를 대라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$\mu=2p-1=0.4$, $\sigma^2=4pq=0.84$. $S_{T_{10}}=10$이므로 Wald: $E[T_{10}]=10/0.4=25$; 2차 항등식: $\mathrm{Var}(T_{10})=\sigma^2a/\mu^3=0.84\cdot10/0.064=131.25$. 꼬리: $P(S_n=a)$의 감쇠율은 $\rho=2\sqrt{pq}\approx0.9165$이고 $\rho^{300}\approx4\times10^{-12}$이므로 300 스텝 절단은 무시할 수 있다.

```python
p_e2, a_e2 = 0.7, 10
mu_e2, s2_e2 = 2 * p_e2 - 1, 4 * p_e2 * (1 - p_e2)
ET_e2, VarT_e2 = a_e2 / mu_e2, s2_e2 * a_e2 / mu_e2**3            # 25, 131.25
walks_e2 = random_walk_paths(10_000, 300, rng, p=p_e2)
T10 = hitting_times(walks_e2, a_e2)
assert np.isfinite(T10).all()
print(mc_estimate(T10), ET_e2)
print(mc_estimate((T10 - T10.mean())**2), VarT_e2)
assert_close(mc_estimate(T10), ET_e2, k=4, name="E2 E[T_10]")
assert_close(mc_estimate((T10 - T10.mean())**2), VarT_e2, k=4, name="E2 Var(T_10)")
```

</details>

### E3. 가정을 깨보기 — 정지시각이 아닌 시각

공정 걸음($\mu=0$)의 100스텝 경로에서 $T^*=\arg\max_{k\le100}S_k$(최댓값이 되는 시각, 여러 개면 아무거나)로 두고 $E[S_{T^*}]$를 시뮬레이션하라. 월드 항등식이 (성립한다면) 예측하는 값은? 실제 값은? 반사 원리로 정확값을 구해 비교하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

Wald가 성립한다면 $E[S_{T^*}]=\mu E[T^*]=0$. 그러나 $S_{T^*}=\max_{k\le100}S_k\ge0$이고 양수일 확률이 있으므로 $E[S_{T^*}]>0$. $T^*$는 경로 전체를 봐야 알 수 있는 시각이므로 정지시각이 아니다 — $\{T^*\ge i\}$가 $X_i$와 독립이 아니어서 증명의 첫 줄이 무너진다. 정확값(반사 원리, 05e에서 재등장): $P(\max_{k\le n}S_k\ge a)=2P(S_n\ge a+1)+P(S_n=a)$이므로
$$E\Big[\max_{k\le100}S_k\Big]=\sum_{a\ge1}\big[2P(S_{100}\ge a+1)+P(S_{100}=a)\big]=7.4987.$$

```python
n_e3 = 100
walks_e3 = random_walk_paths(20_000, n_e3, rng)                     # p = 1/2
S_max = walks_e3.max(axis=1)                                        # S_{T*} = max_{k<=n} S_k
k = np.arange(n_e3 + 1); pS = binom.pmf(k, n_e3, 0.5); S = 2 * k - n_e3
E_max_exact = sum(2 * pS[S >= a + 1].sum() + pS[S == a].sum() for a in range(1, n_e3 + 1))
print(mc_estimate(S_max), "exact", E_max_exact, "Wald would say", 0.0)     # ~7.4987
assert_close(mc_estimate(S_max), E_max_exact, k=4, name="E3 E[max S_k]")
```

</details>

## 7. 정리

1. 월드 항등식 $E[S_T]=\mu E[T]$: i.i.d. 합을 정지시각($E[T]<\infty$)에서 멈추면 '평균 스텝 수 × 스텝당 평균'. 2차: $E[(S_T-\mu T)^2]=\sigma^2E[T]$ ⇒ 도달 문제 $E[T_a]=a/\mu$, $\mathrm{Var}(T_a)=\sigma^2a/\mu^3$.
2. 패턴 대기시간 $E[T_A]=\sum_{k\in\mathrm{ov}(A)}2^k$: 도박꾼 마팅게일에 OST(c)를 적용하면 시각 $T$에 살아 있는 도박꾼(= 겹침)의 자본 합이 $E[T]$가 된다. HTH = 10, HHH = 14, HHT = 8.
3. 같은 값을 KMP 자동자 흡수 사슬(01c)의 $t=N\mathbf 1$로 얻을 수 있다: 마팅게일 풀이 = 기본행렬 풀이 = 시뮬레이션.
4. 자기 겹침이 많은 패턴은 뭉쳐서 나타나므로 첫 등장이 늦다 — 장기 빈도가 같아도 대기시간은 다르다.
5. Penney 게임은 비추이적(THH > HHT > HTT > TTH > THH)이며 Conway 공식 $P(A\text{ before }B)=\frac{B{:}B-B{:}A}{A{:}A-A{:}B+B{:}B-B{:}A}$로 계산된다. '먼저 나오기'와 '평균 대기시간'은 다른 양이다.

**trap 카드**
- Q: HHH와 HTH는 각각 확률 $1/8$로 나오는 패턴인데, 왜 HHH의 기대 대기시간(14)이 HTH(10)보다 긴가?
- A: HHH는 자기 자신과 겹친다(접두부 = 접미부인 길이 1, 2, 3). 한 번 나오면 연달아 나오기 쉬워(HHHH) 같은 장기 빈도 $1/8$을 드문 뭉치로 채운다. 도박꾼 논증에서는 시각 $T$에 도박꾼 3명이 살아 있어 $2+4+8=14$; HTH는 2명($2+8=10$).

**다음 노트북**: 05d — 지금까지는 '고정/정지 시각의 기대값'을 다뤘다. 다음은 $n\to\infty$: 마팅게일 수렴 정리로 폴리아 항아리의 비율이 Beta 확률변수로, 분기 과정의 $Z_n/m^n$이 $W$로, 우도비가 0으로 가는 것을 본다 — 그리고 기대값 1이 극한에서 보존되지 않는 경우를 만난다.

`log/progress.md` 한 줄 템플릿: `- [ ] 05c 월드 항등식과 패턴 대기시간 — 예측 7개 중 __개 자릿수 일치 / E1 __ E2 __ E3 __ / 막힌 곳: __`